In [ ]:
## 1. Import Required Libraries
Install repository dependencies and import helper modules.


!pip install -r /kaggle/working/isan_antispoof/requirements.txt
!pip install dagshub

import os
from pathlib import Path
import pandas as pd
import json


In [ ]:
root = Path('/kaggle/working/IsanAntiSpoof')
print('Repo root:', root)
print('Exists:', root.exists())
print('Kaggle input directory: /kaggle/input')
print('Contents of /kaggle/working:')
print([p.name for p in Path('/kaggle/working').iterdir()])


## 2. Load Dataset and Set Kaggle Paths
Define the repository and Kaggle dataset paths. If you are using a Kaggle dataset, mount it here.


In [ ]:
if root.exists():
    for path in sorted(root.glob('**/*'))[:50]:
        print(path.relative_to(root))
else:
    print('Repo root not found. Clone the repo first.')


## 3. Explore Dataset
Inspect the repository and dataset layout before training.


In [ ]:
print('If raw audio is available in data/raw, run these steps:')
print('python src/data/build_protocol.py')
print('python src/features/extract_all.py feature=lfcc')
print('python src/features/extract_all.py feature=mfcc')
print('python src/features/extract_all.py feature=cqcc')


## 4. Preprocess Audio and Labels
Use the repository scripts to build protocol metadata and extract features from raw audio.


In [ ]:
print('Training is launched with existing repo entrypoint:')
print('python src/training/train.py experiment=e1_baseline')
print('Or use model and feature overrides:')
print('python src/training/train.py experiment=e4_isan_aware model=lcnn feature=mfcc')


## 5. Build the Anti-Spoofing Model
This repository uses `src/training/train.py` and Hydra configuration to build GMM/LCNN/ResNet models.


In [ ]:
## 6. Train the Model
Run the experiment and send metrics to DagsHub MLflow. Replace the placeholders with your DagsHub org and token.


os.environ['DAGSHUB_TOKEN'] = '<your-dagshub-token>'
dagshub_org = '<your-org>'
repo_name = 'IsanAntiSpoof'
mlflow_uri = f'https://dagshub.com/{dagshub_org}/{repo_name}.mlflow'
os.environ['MLFLOW_TRACKING_URI'] = mlflow_uri
print('MLflow URI:', mlflow_uri)

%cd /kaggle/working/IsanAntiSpoof
!python src/training/train.py experiment=e1_baseline mlflow.tracking_uri=$MLFLOW_TRACKING_URI mlflow.experiment_name=isan_antispoof


In [ ]:
results_path = root / 'experiments' / 'results.csv'
print('Results path:', results_path)
if results_path.exists():
    display(pd.read_csv(results_path).tail(10))
else:
    print('No results file found yet.')


## 7. Evaluate Model Performance
Read the aggregated results file produced by the experiment logger.


In [ ]:
checkpoints_dir = root / 'checkpoints'
print('Checkpoint directory exists:', checkpoints_dir.exists())
if checkpoints_dir.exists():
    for path in sorted(checkpoints_dir.glob('**/*'))[:50]:
        print(path.relative_to(root))
else:
    print('No checkpoints saved yet.')
